In [ ]:
import json

def enrollmentevent_fixture(
    transit_agency: int,
    enrollment_flow: int,
    verified_by: str,
    enrollment_datetime: str,
    expiration_datetime: str = None
):
    return {
        "model": "core.enrollmentevent",
        "fields": {
            "transit_agency": transit_agency,
            "enrollment_flow": enrollment_flow,
            "enrollment_method": "digital",
            "verified_by": verified_by,
            "enrollment_datetime": enrollment_datetime,
            "expiration_datetime": expiration_datetime
        }
    }

In [ ]:
AGENCIES = {
    "Monterey-Salinas Transit": {
        "id": 1,
        "flows": {
            "senior": 1,
            "veteran": 2,
            "courtesy_card": 3,
        }
    },
    "Sacramento Regional Transit": {
        "id": 2,
        "flows": {
            "senior": 4,
            "veteran": 7
        }
    },
    "Santa Barbara MTD": {
        "id": 3,
        "flows": {
            "senior": 5,
            "mobility_pass": 6
        }
    },
}

In [ ]:
enrollments = []

with open("metabase-enrollments.json", "r") as e:
    enrollments.extend(json.load(e))

print(len(enrollments))

In [ ]:
events = []

for enrollment in enrollments:
    agency_name = enrollment["Event Properties Transit Agency"]
    agency = AGENCIES.get(agency_name, {})

    flows = agency.get("flows")
    flow_name = enrollment["Event Properties Enrollment Flows"]
    flow = flows.get(flow_name)

    event_time = enrollment.get("Client Event Time")
    event_time = event_time.replace("/", "-").replace(",", "")

    if flow:
        event = enrollmentevent_fixture(
            agency.get("id"),
            flow,
            enrollment.get("Event Properties Eligibility Verifier"),
            event_time
        )

        events.append(event)

with open("events.json", "w") as f:
    json.dump(events, f)